# 🎓 SI-REKTA: Sistem Informasi Rekomendasi Dosen Pembimbing Tugas Akhir Berbasis Machine Learning & NLP

> **Mata Kuliah:** Machine Learning (Semester 5) — Departemen Sistem Informasi  
> **Dosen Pengampu:** Aina Hubby Aziira, M.Eng  
> **Metodologi Pengembangan:** CRISP-ML(Q) (*Cross-Industry Standard Process for Data Mining*)  
> **Repository GitHub:** [https://github.com/accfd/SI-REKTA](https://github.com/accfd/SI-REKTA)  
> **Dokumen Rujukan:**  
> - 📄 Tinjauan Pustaka Sistematis: [`docs/SI-REKTA_SLR.md`](docs/SI-REKTA_SLR.md)  
> - 🗺️ Blueprint Metodologi CRISP-ML(Q): [`docs/SI-REKTA_CRISP_ML_Q.md`](docs/SI-REKTA_CRISP_ML_Q.md)  

---

## 📌 Deskripsi & Tujuan Proyek

Notebook ini merupakan implementasi teknis dari proyek **SI-REKTA** (*Sistem Informasi Rekomendasi Dosen Pembimbing Tugas Akhir*). Masalah utama yang diselesaikan adalah **ketidakcocokan (*mismatch*) antara rencana topik tugas akhir mahasiswa dengan kepakaran riset dosen**, serta **proses penentuan (*plotting*) pembimbing yang memakan waktu lama dan kerap memicu ketimpangan beban kerja dosen (*workload imbalance*)**.

Melalui pendekatan *Natural Language Processing* (NLP) dan *Supervised Machine Learning*, notebook ini memproses data historis 625 judul skripsi dengan karakteristik teks pendek (rata-rata 16 kata) dan ketimpangan kelas yang tajam (satu dosen memiliki 145 bimbingan vs 10–30 bimbingan dosen lain). Solusi komputasional yang diterapkan mengombinasikan ekstraksi fitur **TF-IDF Word (1–2 gram) + Character (3–5 gram)**, klasifikasi **Linear Support Vector Machine (Linear SVM)** berbobot seimbang (*balanced class weighting*), serta **Hybrid Cosine Similarity Gate** terhadap riwayat judul tugas akhir yang pernah dibimbing oleh masing-masing dosen untuk menghasilkan rekomendasi **Top-3 Dosen Pembimbing** yang paling relevan, adil, dan terukur.

Seluruh eksperimen di dalam notebook ini disusun secara sistematis mengacu pada 6 fase **CRISP-ML(Q)** dengan pendekatan **Self-Explaining Notebook**: setiap sel kode selalu diawali dengan landasan teoritis (*"Mengapa tahap ini dilakukan?"*) dan diakhiri dengan evaluasi hasil (*"Interpretasi Output"*).

---

## 📑 Daftar Isi Alur Kerja CRISP-ML(Q):
1. **[Fase 1: Business Understanding](#fase-1-business-understanding)** — Identifikasi masalah, tujuan proyek, dan metrik keberhasilan.
2. **[Fase 2: Data Understanding](#fase-2-data-understanding)** — Pemuatan dataset, Exploratory Data Analysis (EDA), dan audit kualitas data riil.
3. **[Fase 3: Data Preparation](#fase-3-data-preparation)** — Entity canonicalization nama dosen, pembersihan teks, stopword akademik, stemming, dan ekstraksi TF-IDF n-gram.
4. **[Fase 4: Modeling](#fase-4-modeling)** — Pelatihan Baseline Naive Bayes, Main Model Linear SVM (Balanced), dan pembentukan matriks profil Cosine Similarity.
5. **[Fase 5: Evaluation](#fase-5-evaluation)** — Validasi silang Stratified 5-Fold CV, Top-1 & Top-3 Accuracy (Hit@3), Macro-F1 Score, serta analisis keadilan kelas.
6. **[Fase 6: Deployment Preparation](#fase-6-deployment-preparation)** — Serialisasi pipeline mandiri ke `si_rekta_pipeline.joblib` dan simulasi inferensi judul baru.

\n


# <a id="fase-1-business-understanding"></a>1. Fase 1: Business Understanding

> **Referensi Ilmiah:** [`docs/SI-REKTA_SLR.md`](docs/SI-REKTA_SLR.md) Bab 1 & Bab 3

---

### 1.1 Latar Belakang & Identifikasi Masalah Organisasi
Pada Departemen Sistem Informasi, pemilihan dan penentuan dosen pembimbing tugas akhir merupakan proses krusial yang menentukan kelancaran studi mahasiswa. Berdasarkan analisis permasalahan di lapangan:
1. **Ketidaksesuaian Topik (*Topic Mismatch*):** Mahasiswa kerap memilih dosen pembimbing berdasarkan faktor subjektif (popularitas atau kemudahan dihubungi) tanpa mempertimbangkan keselarasan dengan peta jalan (*roadmap*) riset dosen. Hal ini berisiko menyebabkan usulan proposal ditolak oleh komisi tugas akhir atau memerlukan revisi substansial.
2. **Beban Bimbingan Timpang (*Workload Imbalance*):** Penentuan manual oleh koordinator tugas akhir menimbulkan konsentrasi bimbingan ekstrem pada segelintir dosen (misal: 145 bimbingan pada satu dosen vs 10–30 bimbingan pada dosen lain).
3. **Kebutuhan Sistem Pendukung Keputusan:** Diperlukan sistem rekomendasi cerdas yang mampu menganalisis kesesuaian semantik draf judul tugas akhir secara otomatis, objektif, dan terintegrasi ke dalam aplikasi web.

---

### 1.2 Tujuan Proyek (Business & Technical Goals)
* **Tujuan Bisnis:** Membantu mahasiswa menemukan calon pembimbing yang paling relevan secara mandiri serta mempermudah koordinator tugas akhir dalam memverifikasi plotting pembimbing secara adil.
* **Tujuan Teknis ML:** Membangun pipeline NLP dan model klasifikasi multi-kelas berbasis teks judul tugas akhir (bahasa Indonesia) yang mampu memprediksi probabilitas kecocokan dosen dan menyajikan peringkat **Top-3 Dosen Pembimbing** yang paling sesuai.

---

### 1.3 Kriteria Keberhasilan & Indikator Kinerja (KPI)
Berdasarkan sintesis 13 artikel ilmiah bereputasi pada [`docs/SI-REKTA_SLR.md`](docs/SI-REKTA_SLR.md), kriteria keberhasilan model ditetapkan sebagai berikut:
1. **Top-3 Accuracy (Hit Rate @ 3) $\ge$ 80% (0,80):** Dosen pembimbing yang tepat wajib berada dalam 3 rekomendasi teratas sistem minimal pada 80% data uji.
2. **Macro-Averaged F1-Score $\ge$ 70% (0,70):** Menjamin model memiliki presisi dan recall yang adil di seluruh kelas dosen (termasuk dosen kelas minoritas).
3. **Mean Reciprocal Rank (MRR @ 3) $\ge$ 0,60:** Mengukur ketepatan urutan peringkat rekomendasi (posisi 1, 2, atau 3).
4. **Deployability:** Model dapat diekspor (`.joblib`) dan memiliki waktu respons inferensi di bawah 1 detik (*sub-second latency*) untuk integrasi API web.
\n


---

## 🛠️ Pemuatan Pustaka (Library Importation) & Konfigurasi Lingkungan

### 💡 Mengapa Tahap Ini Dilakukan?
Sebelum melakukan manipulasi data dan pemodelan, kita perlu memuat seluruh pustaka (*dependencies*) pendukung yang telah dirancang pada dokumen metodologi [`docs/SI-REKTA_CRISP_DM.md`](docs/SI-REKTA_CRISP_DM.md):
* **Manipulasi & Analisis Data:** `pandas` dan `numpy` untuk operasi tabel data dan manipulasi array numerik.
* **Pembersihan Teks & Regex:** `re`, `string`, dan `collections` untuk prapengolahan string bahasa Indonesia dan analisis frekuensi istilah.
* **Ekstraksi Fitur Teks:** `TfidfVectorizer` dari Scikit-Learn untuk mentransformasi teks judul menjadi representasi numerik berbasis n-gram leksikal dan karakter.
* **Algoritma Machine Learning:** 
  * `MultinomialNB` sebagai model pembanding (*baseline wajib* mengacu pada Sinaga & Kurniawan, 2025).
  * `LinearSVC` dan `CalibratedClassifierCV` sebagai model utama (*main model* berbobot seimbang mengacu pada Bolshakov, 2025).
* **Evaluasi & Pengujian:** `StratifiedKFold`, `train_test_split`, `classification_report`, `confusion_matrix`, dan fungsi metrik ranking.
* **Pengukuran Kemiripan Semantik:** `cosine_similarity` untuk modul *hybrid similarity gate*.
* **Visualisasi Data:** `matplotlib.pyplot` dan `seaborn` untuk pembuatan grafik EDA dan matriks konfusi.
* **Penyimpanan Pipeline:** `joblib` untuk serialisasi model akhir.
\n


In [ ]:
# Pemuatan pustaka standar sistem
import os
import re
import string
import warnings
from collections import Counter

# Pustaka pengolahan data & komputasi numerik
import numpy as np
import pandas as pd

# Pustaka visualisasi data
import matplotlib.pyplot as plt
import seaborn as sns

# Pustaka Scikit-Learn: Text Preprocessing & Feature Extraction
from sklearn.feature_extraction.text import TfidfVectorizer

# Pustaka Scikit-Learn: Modeling & Calibration
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.pipeline import Pipeline

# Pustaka Scikit-Learn: Evaluation & Validation
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# Pustaka serialisasi model
import joblib

# Konfigurasi visualisasi & warning
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['axes.edgecolor'] = '#cccccc'
plt.rcParams['axes.linewidth'] = 0.8

print("✅ Seluruh pustaka berhasil dimuat dengan sukses!")
\n

### 🔍 Interpretasi Hasil Setup Lingkungan
Seluruh pustaka pendukung (*stack*) ilmu data, pemrosesan teks, pemodelan linier berbobot seimbang, dan evaluasi telah berhasil dimuat tanpa kendala dependensi. Lingkungan komputasi kini siap untuk melanjutkan ke **Fase 2: Data Understanding** (pemuatan dataset `docs/Judul_TA.csv` dan analisis data eksploratif).
\n


# <a id="fase-2-data-understanding"></a>2. Fase 2: Data Understanding & Exploratory Data Analysis (EDA)

> **Tahap Metodologi:** CRISP-ML(Q) — Fase 2: Data Understanding  
> **Tujuan Utama:** Memahami karakteristik data mentah, mengidentifikasi kualitas data (nilai kosong dan placeholder), mendeteksi ketimpangan kelas dosen, serta menganalisis panjang teks judul skripsi.

---

## 2.1 Pemuatan Data (Data Loading) & Pemeriksaan Struktur Awal

### 💡 Mengapa Tahap Ini Dilakukan?
Sebelum melakukan pengolahan data, kita perlu memastikan data mentah dari file `docs/Judul_TA.csv` dapat dibaca dengan benar oleh sistem. Karena data diekspor dari sistem informasi kampus di Indonesia, pemisah kolom menggunakan titik koma (`;`). Tahap ini memeriksa dimensi tabel, tipe data setiap kolom, serta menampilkan sampel data teratas agar kita mengetahui kolom apa saja yang tersedia.
\n

In [ ]:
# Memuat dataset tugas akhir menggunakan pandas
csv_path = os.path.join("docs", "Judul_TA.csv")
df_raw = pd.read_csv(csv_path, sep=';', encoding='utf-8')

# Menampilkan dimensi dan 5 baris pertama data
print(f"📊 Total baris data: {df_raw.shape[0]} baris")
print(f"📋 Total kolom: {df_raw.shape[1]} kolom\n")

df_raw.head()
\n

### 🔍 Interpretasi Hasil Pemuatan Data
Dataset berhasil dimuat dengan total **625 baris data tugas akhir** dan **8 kolom**:
* `no`: Nomor urut pencatatan.
* `judul_ta`: Teks usulan judul tugas akhir mahasiswa (fitur utama input teks).
* `nama_mahasiswa`: Nama lengkap mahasiswa.
* `nim`: Nomor Induk Mahasiswa.
* `tahun`: Tahun kelulusan/pengajuan tugas akhir (rentang 2014–2026).
* `pembimbing_1`: Nama dosen pembimbing utama (target klasifikasi utama).
* `pembimbing_2`: Nama dosen pembimbing pendamping (bila ada).
* `poster`: Link atau status file poster skripsi.
\n

---

## 2.2 Audit Kualitas Data: Missing Values & Nilai Placeholder

### 💡 Mengapa Tahap Ini Dilakukan?
Dalam data dunia nyata di instansi kampus, data kosong sering kali tidak tercatat sebagai *null* (kosong murni), melainkan diisi dengan tanda strip (`'-'`). Kita perlu mendeteksi kedua jenis nilai kosong ini untuk menentukan kolom mana yang layak dipertahankan dan kolom mana yang harus dibuang (*drop*).
\n

In [ ]:
# 1. Cek missing value standar (Null / NaN)
null_counts = df_raw.isnull().sum()

# 2. Cek nilai placeholder tanda strip ('-')
dash_counts = (df_raw == '-').sum()

# Menggabungkan hasil pengecekan ke dalam satu tabel ringkasan
data_quality = pd.DataFrame({
    'Tipe Data': df_raw.dtypes,
    'Jumlah Null': null_counts,
    'Persentase Null (%)': (null_counts / len(df_raw)) * 100,
    'Jumlah Strip (-)': dash_counts,
    'Persentase Strip (%)': (dash_counts / len(df_raw)) * 100
})

data_quality
\n

### 🔍 Interpretasi Kualitas Data
Hasil audit kualitas data menunjukkan dua temuan penting:
1. **Kolom `poster` tidak informatif**: Memiliki **86,88% nilai placeholder ('-')**. Kolom ini tidak memiliki nilai analitis untuk rekomendasi teks dan akan dibuang pada tahap Data Preparation.
2. **Kolom `pembimbing_2` mayoritas kosong**: Memiliki **73,12% nilai placeholder ('-')**. Ini menunjukkan bahwa mayoritas skripsi di jurusan hanya memiliki 1 pembimbing utama. Oleh karena itu, target prediksi utama difokuskan pada kolom `pembimbing_1`.
3. **Kolom `judul_ta` dan `pembimbing_1` bersih dari nilai kosong**: Memiliki 0% null dan 0% strip, sehingga 100% data judul siap diolah lebih lanjut.
\n

---

## 2.3 Analisis Sebaran Dosen Pembimbing (Class Imbalance Analysis)

### 💡 Mengapa Tahap Ini Dilakukan?
Untuk melihat apakah pembagian bimbingan antar-dosen sudah merata atau timpang. Jika ada satu dosen yang membimbing jauh lebih banyak mahasiswa dibanding dosen lain, model machine learning biasa akan menjadi bias (selalu menebak dosen populer tersebut). Tahap ini juga memeriksa variasi ejaan nama dosen yang tidak konsisten.
\n

In [ ]:
# Menghitung frekuensi bimbingan per nama dosen mentah
dosen_counts = df_raw['pembimbing_1'].value_counts()

print(f"👥 Jumlah penulisan nama dosen unik (mentah): {len(dosen_counts)} variasi entitas\n")

# Visualisasi 15 Dosen dengan Bimbingan Terbanyak (Data Mentah)
plt.figure(figsize=(12, 7))
top_15_dosen = dosen_counts.head(15)
ax = sns.barplot(x=top_15_dosen.values, y=top_15_dosen.index, palette='Blues_r')

plt.title('Top 15 Dosen Pembimbing Berdasarkan Jumlah Bimbingan (Data Mentah)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Jumlah Judul Tugas Akhir yang Dibimbing', fontsize=12)
plt.ylabel('Nama Dosen Pembimbing', fontsize=12)

# Menambahkan label angka di ujung bar
for i, v in enumerate(top_15_dosen.values):
    ax.text(v + 1.5, i, str(v), color='black', va='center', fontweight='bold')

plt.xlim(0, max(top_15_dosen.values) + 15)
plt.tight_layout()
plt.show()

# Contoh deteksi inkonsistensi nama dosen (Studi Kasus: Prof. Surya Afnarius)
print("🔍 Contoh Inkonsistensi Penulisan Nama Dosen (Prof. Surya Afnarius):")
surya_variations = df_raw[df_raw['pembimbing_1'].str.contains('Surya', na=False)]['pembimbing_1'].value_counts()
for name, count in surya_variations.items():
    print(f"  • '{name}' : muncul {count} kali")
\n

### 🔍 Interpretasi Sebaran Dosen & Inkonsistensi Nama
Grafik dan statistik di atas membuktikan dua fakta lapangan yang sangat krusial:
1. **Ketimpangan Kelas Ekstrem (*Extreme Class Imbalance*):**
   * Pak Ricky Akbar menempati urutan pertama dengan **145 bimbingan**, disusul Pak Husnil Kamil (**93 bimbingan**), dan Pak Haris Suryamen (**67 bimbingan**).
   * Sementara itu, terdapat banyak dosen lain yang hanya membimbing antara **10 hingga 20 mahasiswa**.
   * *Konsekuensi Metodologis:* Kita **wajib** menggunakan pembobotan kelas seimbang (`class_weight='balanced'`) pada Linear SVM agar model tidak selalu condong menebak dosen mayoritas.
2. **Inkonsistensi Format String Nama Dosen:**
   * Nama *Prof. Surya Afnarius* tertulis dalam beberapa format berbeda: `'Prof. Surya Afnarius, Ph.D'` (59x), `'Prof. Surya Afnarius, PhD'` (11x), `'Prof.Surya Afnarius,Ph.D'` (10x tanpa spasi), dan `'Prof. Surya Afnarius, P.Hd'` (4x).
   * *Konsekuensi Metodologis:* Pada Fase Data Preparation, kita **wajib** membuat kamus penyeragaman nama (*canonicalization*) agar sistem mengenali mereka sebagai satu dosen yang sama.
\n

---

## 2.4 Analisis Karakteristik Teks Judul Tugas Akhir

### 💡 Mengapa Tahap Ini Dilakukan?
Mengetahui panjang kata dari judul skripsi dan kata-kata apa saja yang paling sering muncul. Informasi ini menentukan strategi ekstraksi fitur: teks pendek (10–20 kata) sangat cocok diproses menggunakan TF-IDF N-Gram karena lebih cepat, akurat, dan tidak mudah *overfitting* dibanding model deep learning berukuran besar.
\n

In [ ]:
# Menghitung panjang kata pada setiap judul tugas akhir
df_raw['panjang_kata'] = df_raw['judul_ta'].astype(str).apply(lambda x: len(x.split()))

# Statistik deskriptif panjang kata
stats_panjang = df_raw['panjang_kata'].describe()
print("📏 Statistik Panjang Judul (Jumlah Kata):")
print(f"  • Rata-rata : {stats_panjang['mean']:.1f} kata")
print(f"  • Terpendek  : {int(stats_panjang['min'])} kata")
print(f"  • Terpanjang : {int(stats_panjang['max'])} kata")
print(f"  • Standar Deviasi : {stats_panjang['std']:.1f} kata\n")

# Visualisasi Distribusi Panjang Kata
plt.figure(figsize=(10, 4))
sns.histplot(df_raw['panjang_kata'], bins=20, kde=True, color='#2b5c8f')
plt.axvline(stats_panjang['mean'], color='red', linestyle='--', label=f'Rata-rata: {stats_panjang['mean']:.1f} kata')
plt.title('Distribusi Panjang Kata pada Judul Tugas Akhir', fontsize=13, fontweight='bold', pad=12)
plt.xlabel('Jumlah Kata per Judul', fontsize=11)
plt.ylabel('Frekuensi Judul', fontsize=11)
plt.legend()
plt.tight_layout()
plt.show()

# Analisis Top-20 Kata Paling Sering Muncul (Tanpa Stopwords Umum)
stop_umum = {
    'dan', 'di', 'pada', 'untuk', 'dengan', 'menggunakan', 'metode', 'berbasis',
    'dalam', 'terhadap', 'dari', 'ke', 'yang', 'secara', 'studi', 'kasus'
}

all_words = []
for judul in df_raw['judul_ta'].dropna():
    words = re.findall(r'\b[a-zA-Z]{3,}\b', judul.lower())
    for w in words:
        if w not in stop_umum:
            all_words.append(w)

word_counts = Counter(all_words).most_common(20)
words_df = pd.DataFrame(word_counts, columns=['Kata Kunci Domain', 'Frekuensi'])

plt.figure(figsize=(12, 6))
ax = sns.barplot(x='Frekuensi', y='Kata Kunci Domain', data=words_df, palette='viridis')
plt.title('Top 20 Kata Kunci Domain Paling Dominan pada Judul Tugas Akhir', fontsize=13, fontweight='bold', pad=12)
plt.xlabel('Jumlah Kemunculan', fontsize=11)
plt.ylabel('Kata Kunci', fontsize=11)

for i, v in enumerate(words_df['Frekuensi']):
    ax.text(v + 3, i, str(v), color='black', va='center', fontweight='bold', fontsize=10)

plt.xlim(0, max(words_df['Frekuensi']) + 30)
plt.tight_layout()
plt.show()
\n

### 🔍 Interpretasi Karakteristik Teks Judul
1. **Konfirmasi Teks Pendek (*Short-Text*):**
   * Rata-rata panjang judul tugas akhir adalah **16 kata**, dengan mayoritas berkisar antara **13 hingga 19 kata**.
   * Judul terpendek memiliki 6 kata dan terpanjang 29 kata.
   * *Kesimpulan Metodologis:* Data ini membuktikan bahwa teks judul berukuran ringkas, sehingga pendekatan leksikal **TF-IDF Word (1–2 gram) dan Char (3–5 gram)** adalah solusi terbaik yang hemat komputasi dan tidak rentan *overfitting*.
2. **Pola Topik Sistem Informasi yang Terbentuk:**
   * Kata-kata seperti *informasi (310)*, *web (224)*, *mobile (129)*, *pendukung keputusan / SPK (98)*, *pengelolaan (86)*, *business intelligence (71)*, dan *geografis / GIS (56)* mencerminkan klaster kepakaran dosen di jurusan.
   * Kata-kata generik formal skripsi (seperti *pembangunan, sistem, aplikasi, implementasi, perancangan*) wajib dibersihkan pada tahap prapengolahan agar bobot TF-IDF berfokus murni pada kata kunci kepakaran.
\n